# Bank Settlement Reconciliation - May & June 2026

Matching our own payment records against the bank settlement statement for two months,
and carrying May's unfinished items into June.

**What this notebook produces**

| File | What it is |
| --- | --- |
| `reconciliation_output.csv` | Every group of records, with the result and the difference |
| `exception_report.csv` | Everything that needs a person to act, with what to do |
| `backlog_report.csv` | May items still open at 31 May, and whether June cleared them |
| `input_validation_report.csv` | Every input check, passed or failed |
| `summary_control_report.csv` | The headline numbers for both months |
| `reconciliation_dashboard.html` | A one page summary you can open in a browser |

**The words we use in the output files**

The brief asks for four exact statuses and two exact backlog labels. We use those words,
just in title case instead of shouting caps, plus one 1:1/1:N/N:1/N:M gloss on cardinality:

| Word | What it means |
| --- | --- |
| Matched | Our amount and the bank amount agree |
| Partial Match | We found the credit, but the amount is not the same |
| Open | No bank credit found for this payment |
| Exception | Something is wrong and a person has to look at it |
| Likely Settlement Lag | A May item still open, but young enough to just be running late |
| Genuine Exception | A May item still open and old enough that someone should chase it |
| Group | One unit of work: one or more payments with the credits that settled them |
| Difference | Our amount minus the bank amount |
| Days Old | How many days from the payment date to the end of that month |

Every column heading in the CSV files is written in plain English, so no code list is needed
to read them.

**How to run it**

Put the four input files in `data/`, then run every cell from top to bottom. The six output
files are written to `output/`.

## 1. Setup

We only need `pandas` (for reading the CSV files), `re` (to pull references out of the
bank narration text) and `os` (to create the output folder).

Two settings matter:

* **`TOLERANCE = 1.00`** - if the ledger and the bank differ by a rupee or less we treat it
  as rounding and call it matched. Anything bigger is a real difference.
* **`LAG_DAYS = 5`** - if a transaction is still unpaid at month end but is 5 days old or
  less, we assume the bank is simply running late. Older than that and we treat it as a
  problem to investigate.

In [1]:
import pandas as pd
import re
import os

DATA_DIR = "data"
OUT_DIR = "output"
TOLERANCE = 1.00          # rupees. A gap this small is treated as rounding, not a break.
LAG_DAYS = 5              # open items younger than this are "settlement lag", older are exceptions

MAY_END = pd.Timestamp("2026-05-31")
JUN_END = pd.Timestamp("2026-06-30")

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

print("Setup done.")

Setup done.


## 2. Read the files and check them

Before matching anything we check the inputs and write every finding into
`input_validation_report.csv`. Nothing is silently dropped.

For each file we check that it can be read, that the columns we need are there, that the
dates and amounts are real values, that no row is missing its id, and that ids are unique.
For the ledger we also check the currency, the status and whether a payment reference is
used twice. For the statement we check every row is a credit.

Each finding gets a severity: **PASS** (fine), **WARNING** (odd but usable) or **ERROR**
(must be fixed).

In [2]:
validation_rows = []


def log_check(file_name, check, severity, detail, record_id=""):
    validation_rows.append({
        "file": file_name,
        "check": check,
        "severity": severity,
        "record_id": record_id,
        "issue_detail": detail,
    })


INTERNAL_COLUMNS = ["txn_id", "txn_date", "channel", "merchant_id", "txn_type",
                    "amount", "currency", "payment_ref", "batch_id", "status"]
BANK_COLUMNS = ["line_id", "value_date", "narration", "dr_cr", "amount", "bank_ref"]


def read_and_check(file_name, needed_columns, id_column, date_column):
    """Read one CSV and write every problem found into the validation report."""
    path = os.path.join(DATA_DIR, file_name)

    try:
        df = pd.read_csv(path)
        log_check(file_name, "FILE_READABLE", "PASS", str(len(df)) + " rows read")
    except Exception as e:
        log_check(file_name, "FILE_READABLE", "ERROR", "Could not read file: " + str(e))
        return pd.DataFrame(columns=needed_columns)

    missing = []
    for col in needed_columns:
        if col not in df.columns:
            missing.append(col)
    if missing:
        log_check(file_name, "REQUIRED_COLUMNS", "ERROR", "Missing columns: " + ", ".join(missing))
        return pd.DataFrame(columns=needed_columns)
    log_check(file_name, "REQUIRED_COLUMNS", "PASS", "All required columns present")

    # dates
    parsed = pd.to_datetime(df[date_column], errors="coerce")
    bad_dates = df[parsed.isna()]
    if len(bad_dates) > 0:
        for rid in bad_dates[id_column]:
            log_check(file_name, "VALID_DATE", "ERROR", "Date could not be read", rid)
    else:
        log_check(file_name, "VALID_DATE", "PASS", "All dates valid")
    df[date_column] = parsed

    # amounts
    amounts = pd.to_numeric(df["amount"], errors="coerce")
    bad_amounts = df[amounts.isna()]
    if len(bad_amounts) > 0:
        for rid in bad_amounts[id_column]:
            log_check(file_name, "VALID_AMOUNT", "ERROR", "Amount is missing or not a number", rid)
    else:
        log_check(file_name, "VALID_AMOUNT", "PASS", "All amounts numeric")
    df["amount"] = amounts.fillna(0).round(2)

    # identifiers
    blank_ids = df[df[id_column].isna() | (df[id_column].astype(str).str.strip() == "")]
    if len(blank_ids) > 0:
        log_check(file_name, "ID_PRESENT", "ERROR", str(len(blank_ids)) + " rows have no " + id_column)
    else:
        log_check(file_name, "ID_PRESENT", "PASS", "All rows have an id")

    dup_ids = df[df[id_column].duplicated()]
    if len(dup_ids) > 0:
        for rid in dup_ids[id_column]:
            log_check(file_name, "UNIQUE_ID", "ERROR", "Duplicated id", rid)
    else:
        log_check(file_name, "UNIQUE_ID", "PASS", "All ids unique")

    # file-specific checks
    if id_column == "txn_id":
        blank_ref = df[df["payment_ref"].isna()]
        if len(blank_ref) > 0:
            log_check(file_name, "PAYMENT_REF_PRESENT", "ERROR",
                      str(len(blank_ref)) + " rows have no payment_ref")
        else:
            log_check(file_name, "PAYMENT_REF_PRESENT", "PASS", "All rows have a payment_ref")

        not_inr = df[df["currency"] != "INR"]
        if len(not_inr) > 0:
            log_check(file_name, "CURRENCY", "WARNING", str(len(not_inr)) + " rows are not INR")
        else:
            log_check(file_name, "CURRENCY", "PASS", "All rows are INR")

        not_success = df[df["status"] != "SUCCESS"]
        if len(not_success) > 0:
            log_check(file_name, "STATUS", "WARNING",
                      str(len(not_success)) + " rows are not SUCCESS and were excluded")
        else:
            log_check(file_name, "STATUS", "PASS", "All rows are SUCCESS")

        repeated_ref = df[df["payment_ref"].duplicated(keep=False)]
        if len(repeated_ref) > 0:
            for rid in repeated_ref["txn_id"]:
                log_check(file_name, "DUPLICATE_PAYMENT_REF", "WARNING",
                          "payment_ref is used by more than one transaction", rid)
        else:
            log_check(file_name, "DUPLICATE_PAYMENT_REF", "PASS", "All payment_refs unique")
    else:
        not_credit = df[df["dr_cr"] != "CR"]
        if len(not_credit) > 0:
            log_check(file_name, "CREDIT_ONLY", "WARNING",
                      str(len(not_credit)) + " rows are not credits and were excluded")
        else:
            log_check(file_name, "CREDIT_ONLY", "PASS", "All rows are credits")

    return df


may_internal = read_and_check("internal_txns_may2026.csv", INTERNAL_COLUMNS, "txn_id", "txn_date")
may_bank = read_and_check("bank_stmt_may2026.csv", BANK_COLUMNS, "line_id", "value_date")
jun_internal = read_and_check("internal_txns_jun2026.csv", INTERNAL_COLUMNS, "txn_id", "txn_date")
jun_bank = read_and_check("bank_stmt_jun2026.csv", BANK_COLUMNS, "line_id", "value_date")

# keep only rows we are supposed to reconcile
may_internal = may_internal[may_internal["status"] == "SUCCESS"].copy()
jun_internal = jun_internal[jun_internal["status"] == "SUCCESS"].copy()
may_bank = may_bank[may_bank["dr_cr"] == "CR"].copy()
jun_bank = jun_bank[jun_bank["dr_cr"] == "CR"].copy()

print("May:", len(may_internal), "internal /", len(may_bank), "bank")
print("Jun:", len(jun_internal), "internal /", len(jun_bank), "bank")

May: 328 internal / 252 bank
Jun: 288 internal / 250 bank


## 3. Clean up the bank narration

This is the step that does most of the work. The bank writes the reference in the narration
text, but not always neatly. Real examples from the data:

| What the bank wrote | The problem |
| --- | --- |
| `SETTLE/PRMA Y95565484/PAYOUT` | a stray space in the middle |
| `SETTLE/prmay85386435/PAYOUT` | lower case |
| `SETTLE/NEFTPRMAY58438711/PAYOUT` | channel name glued to the front |
| `SETTLE/PRMAY832847/PAYOUT` | the reference is cut short |

Making everything upper case and deleting every space fixes the first three. For the
fourth we look for references in our ledger that *start with* what the bank wrote - if
exactly one does, that is the answer; if more than one does, we leave it alone rather than
guess.

The same idea is used to pull out batch numbers, declared bank charges, part numbers and
net-settlement windows.

In [3]:
def tidy(text):
    """Upper case and remove every space, so 'PRMA Y123' becomes 'PRMAY123'."""
    return str(text).upper().replace(" ", "")


def find_reference(narration):
    """Pull the PRMAY.../PRJUN... reference out of a narration. Returns '' if there is none."""
    found = re.search(r"PR(?:MAY|JUN)\d+", tidy(narration))
    if found:
        return found.group(0)
    return ""


def expand_reference(short_ref, all_refs):
    """Some narrations carry a chopped-off reference. If exactly one real reference
    starts with it, use that one. Otherwise leave it as-is."""
    if short_ref == "" or short_ref in all_refs:
        return short_ref
    starts_with = [r for r in all_refs if r.startswith(short_ref)]
    if len(starts_with) == 1:
        return starts_with[0]
    return short_ref


def find_batch(narration):
    found = re.search(r"BATCH(?:MAY|JUN)\d+", tidy(narration))
    if found:
        return found.group(0)
    return ""


def find_charges(narration):
    """'NET OF CHGS 330.26' means the bank kept 330.26. Add it back before comparing."""
    found = re.search(r"NETOFCHGS([\d.]+)", tidy(narration))
    if found:
        return float(found.group(1))
    return 0.0


def find_part_number(narration):
    """'PART SETTLEMENT PRMAY48951595-P2' -> 2"""
    found = re.search(r"-P(\d+)", tidy(narration))
    if found:
        return int(found.group(1))
    return 0


def find_net_window(narration):
    """'NET SETTLEMENT M1016 2026-05-19 1/2' -> ('M1016', '2026-05-19')"""
    found = re.search(r"NETSETTLEMENTM(\d+)(\d{4}-\d{2}-\d{2})", tidy(narration))
    if found:
        return "M" + found.group(1), found.group(2)
    return "", ""


# quick check that the cleaning works
print(find_reference("SETTLE/PRMA Y95565484/PAYOUT"))
print(find_reference("SETTLE/prmay85386435/PAYOUT"))
print(find_reference("SETTLE/NEFTPRMAY58438711/PAYOUT"))
print(find_batch("SETTLEMENT BATCHMAY001 NET OF CHGS 330.26"), find_charges("SETTLEMENT BATCHMAY001 NET OF CHGS 330.26"))
print(find_net_window("NET SETTLEMENT M1016 2026-05-19 1/2"))

PRMAY95565484
PRMAY85386435
PRMAY58438711
BATCHMAY001 330.26
('M1016', '2026-05-19')


## 4. The matching engine

One function reconciles one month. It puts both sides into plain dictionaries, then runs
six passes **in order, most specific first**. Every pass has to go through `make_group()`.

`make_group()` is the control that prevents double counting. Before it creates a group it
checks that none of the records are already in `used_txns` or `used_lines`. If even one of
them is, it refuses and nothing changes. So a record can only ever land in one group.

**The passes:**

1. **Internal self-netting** - a SALE and a REVERSAL on the same reference that cancel out.
   Removed first so they never look unpaid.
2. **Batch settlements (N:1)** - many transactions paid as one credit, e.g.
   `SETTLEMENT BATCHMAY006`. Where the narration says `NET OF CHGS 330.26` we add that fee
   back before comparing, which turns an apparent shortfall into an exact tie.
3. **Part settlements (1:N)** - one transaction paid in instalments, e.g. `-P1`, `-P2`, `-P3`.
   All parts are collected together first, so no single instalment can claim the whole amount.
4. **Payment reference (1:1)** - the main pass. Credits are taken oldest first, but only
   while they do not overshoot what is owed. Anything left over on the same reference is a
   **repeat credit** and becomes an exception. If the credits fall short, the group stays
   `PARTIAL_MATCH` - the missing money stays visible.
5. **Net settlement sweeps (N:M)** - `NET SETTLEMENT M1016 2026-05-19` means the bank netted
   a whole day for one merchant.
6. **Whatever is left** - transactions with no credit become `OPEN`; credits with no
   transaction become orphan credits.

**Why pass 5 runs last.** A net sweep grabs everything for a merchant on a date. If it ran
early it would swallow transactions that actually have their own credit. For example
`TXNMAY00104` sits inside the M1010 net-settlement window but is really paid by its own line
`BLMAY00104`. Running the sweep after reference matching means it can only pick up the
genuine leftovers.

The function ends with two assertions: every transaction and every bank credit must appear
in exactly one group. If that is ever untrue the notebook stops with an error rather than
quietly producing wrong numbers.

In [ ]:
def bucket_of(days):
    if days <= 7:
        return "0-7d"
    if days <= 15:
        return "8-15d"
    if days <= 30:
        return "16-30d"
    if days <= 60:
        return "31-60d"
    return "60d+"


def reconcile(internal_df, bank_df, period_name, period_end):
    """Match one period. Returns a list of match groups (plain dictionaries)."""

    # ---- put both sides into simple dictionaries keyed by id -------------
    txns = {}
    for _, r in internal_df.iterrows():
        txns[r["txn_id"]] = {
            "txn_id": r["txn_id"], "date": r["txn_date"], "amount": round(float(r["amount"]), 2),
            "ref": str(r["payment_ref"]).strip().upper(), "merchant": r["merchant_id"],
            "channel": r["channel"], "type": r["txn_type"],
            "batch": "" if pd.isna(r["batch_id"]) else str(r["batch_id"]).strip().upper(),
            "origin": r["origin_period"],
        }

    lines = {}
    known_refs = set(t["ref"] for t in txns.values())
    for _, r in bank_df.iterrows():
        raw_ref = find_reference(r["narration"])
        lines[r["line_id"]] = {
            "line_id": r["line_id"], "date": r["value_date"], "amount": round(float(r["amount"]), 2),
            "narration": r["narration"], "bank_ref": r["bank_ref"],
            "ref": expand_reference(raw_ref, known_refs),
            "batch": find_batch(r["narration"]), "charges": find_charges(r["narration"]),
            "part": find_part_number(r["narration"]),
        }

    used_txns = set()
    used_lines = set()
    groups = []
    counter = [0]

    def free_txns():
        return [t for t in txns.values() if t["txn_id"] not in used_txns]

    def free_lines():
        return [b for b in lines.values() if b["line_id"] not in used_lines]

    def cardinality(n_txn, n_line):
        left = "1" if n_txn == 1 else "N"
        right = "1" if n_line == 1 else "N"
        if n_txn == 0:
            left = "0"
        if n_line == 0:
            right = "0"
        if left == "N" and right == "N":
            return "N:M"
        return left + ":" + right

    def make_group(txn_ids, line_ids, method, comment,
                   status=None, category="", charges=0.0):
        """Create one match group. Refuses if any record was already used -
        this is what stops a record being matched twice."""
        for t in txn_ids:
            if t in used_txns:
                return None
        for b in line_ids:
            if b in used_lines:
                return None

        internal_amount = round(sum(txns[t]["amount"] for t in txn_ids), 2)
        bank_amount = round(sum(lines[b]["amount"] for b in line_ids) + charges, 2)
        variance = round(internal_amount - bank_amount, 2)

        if status is None:
            if len(line_ids) == 0:
                status = "OPEN"
            elif abs(variance) <= TOLERANCE:
                status = "MATCHED"
            else:
                status = "PARTIAL_MATCH"

        counter[0] += 1
        dates = [txns[t]["date"] for t in txn_ids] + [lines[b]["date"] for b in line_ids]
        start_date = min(dates)
        age = (period_end - start_date).days

        group = {
            "period": period_name,
            "match_group_id": period_name + "-G" + str(counter[0]).zfill(4),
            "internal_txn_ids": ",".join(sorted(txn_ids)),
            "bank_line_ids": ",".join(sorted(line_ids)),
            "payment_refs": ",".join(sorted(set(txns[t]["ref"] for t in txn_ids))),
            "bank_refs": ",".join(sorted(lines[b]["bank_ref"] for b in line_ids)),
            "merchant_id": ",".join(sorted(set(txns[t]["merchant"] for t in txn_ids))),
            "cardinality": cardinality(len(txn_ids), len(line_ids)),
            "internal_amount": internal_amount,
            "internal_gross_amount": round(sum(abs(txns[t]["amount"]) for t in txn_ids), 2),
            "bank_amount": bank_amount,
            "variance": variance,
            "status": status,
            "match_method": method,
            "exception_category": category,
            "start_date": str(start_date.date()),
            "ageing_days": age,
            "ageing_bucket": bucket_of(age),
            "comments": comment,
            "origin": ",".join(sorted(set(txns[t]["origin"] for t in txn_ids))) if txn_ids else "",
        }
        groups.append(group)
        for t in txn_ids:
            used_txns.add(t)
        for b in line_ids:
            used_lines.add(b)
        return group

    # ---- PASS 1: internal sale + reversal that cancel each other out -----
    by_ref = {}
    for t in txns.values():
        by_ref.setdefault(t["ref"], []).append(t)
    for ref, group_txns in by_ref.items():
        if len(group_txns) > 1 and abs(sum(t["amount"] for t in group_txns)) <= TOLERANCE:
            make_group([t["txn_id"] for t in group_txns], [],
                       "1. internal self-netting", "Sale and refund on the same reference cancel out. "
                       "No bank credit is due.",
                       status="EXCEPTION", category="internal_self_netting_pair")

    # ---- PASS 2: batch settlements (many transactions, one credit) ------
    for b in sorted(free_lines(), key=lambda x: x["line_id"]):
        if b["batch"] == "":
            continue
        members = [t["txn_id"] for t in free_txns() if t["batch"] == b["batch"]]
        if not members:
            continue
        note = "Bank paid batch " + b["batch"] + " as one credit."
        if b["charges"] > 0:
            note = note + " Bank fees of " + str(b["charges"]) + " were added back."
        make_group(members, [b["line_id"]], "2. batch id", note, charges=b["charges"])

    # ---- PASS 3: part settlements (one transaction, several credits) ----
    part_refs = {}
    for b in free_lines():
        if b["part"] > 0 and b["ref"] != "":
            part_refs.setdefault(b["ref"], []).append(b)
    for ref, parts in part_refs.items():
        members = [t["txn_id"] for t in free_txns() if t["ref"] == ref]
        if not members:
            continue
        parts = sorted(parts, key=lambda x: x["part"])
        make_group(members, [p["line_id"] for p in parts],
                   "3. part settlement",
                   "Bank paid this in " + str(len(parts)) + " parts.")

    # ---- PASS 4: reference match (covers 1:1, repeats and short pays) ---
    ref_lines = {}
    for b in free_lines():
        if b["ref"] != "":
            ref_lines.setdefault(b["ref"], []).append(b)

    for ref in sorted(ref_lines.keys()):
        candidates = [t for t in free_txns() if t["ref"] == ref]
        if len(candidates) != 1:
            continue
        txn = candidates[0]
        candidate_lines = sorted(ref_lines[ref], key=lambda x: (x["date"], x["line_id"]))

        # take credits oldest first, but only while they do not overshoot the amount owed
        taken = []
        running = 0.0
        extra = []
        for b in candidate_lines:
            if running + b["amount"] <= txn["amount"] + TOLERANCE:
                taken.append(b)
                running = round(running + b["amount"], 2)
            else:
                extra.append(b)

        if not taken:                       # every credit overshoots -> take the first one only
            taken = [candidate_lines[0]]
            extra = candidate_lines[1:]

        note = "Bank narration shows reference " + ref + "."
        if len(taken) > 1:
            note = note + " Paid across " + str(len(taken)) + " credits."
        if tidy(taken[0]["narration"]).find("PREVCYCLE") >= 0:
            note = note + " Narration marks it as a previous cycle."
        g = make_group([txn["txn_id"]], [b["line_id"] for b in taken], "4. payment reference", note)

        if g is not None and g["status"] == "PARTIAL_MATCH" and g["variance"] > 0:
            g["exception_category"] = "short_settlement"
            g["comments"] = note + " Bank paid " + str(round(g["variance"], 2)) + " less than our books."

        # anything left over on the same reference is a repeat credit
        for b in extra:
            make_group([], [b["line_id"]], "4. payment reference",
                       "Reference " + ref + " was already paid in full. "
                       "This credit is a repeat.",
                       status="EXCEPTION", category="duplicate_credit")

    # ---- PASS 5: net settlement sweeps (many to many) -------------------
    windows = {}
    for b in free_lines():
        merchant, day = find_net_window(b["narration"])
        if merchant != "":
            windows.setdefault((merchant, day), []).append(b)
    for (merchant, day), window_lines in windows.items():
        members = [t["txn_id"] for t in free_txns()
                   if t["merchant"] == merchant and str(t["date"].date()) == day]
        if not members:
            continue
        make_group(members, [b["line_id"] for b in window_lines],
                   "5. net settlement sweep",
                   "Bank rolled " + str(len(members)) + " payments for merchant " + merchant +
                   " dated " + day + " into " + str(len(window_lines)) + " credit(s).")

    # ---- PASS 6: nothing left to match ---------------------------------
    for t in sorted(free_txns(), key=lambda x: x["txn_id"]):
        make_group([t["txn_id"]], [], "6. unmatched",
                   "No bank credit found for this payment.")

    for b in sorted(free_lines(), key=lambda x: x["line_id"]):
        make_group([], [b["line_id"]], "6. unmatched",
                   "This credit does not match any payment in our books.",
                   status="EXCEPTION", category="orphan_credit")

    # ---- control check: every record used exactly once ------------------
    assert len(used_txns) == len(txns), "Some transactions were not placed in a group"
    assert len(used_lines) == len(lines), "Some bank credits were not placed in a group"

    return groups


print("Engine ready.")

## 5. Task 1 - reconcile May

In [5]:
may_internal["origin_period"] = "MAY"
may_groups = reconcile(may_internal, may_bank, "MAY", MAY_END)
may_out = pd.DataFrame(may_groups)
print(may_out.groupby("status")["match_group_id"].count())
print(may_out["cardinality"].value_counts())

status
EXCEPTION         15
MATCHED          226
OPEN              30
PARTIAL_MATCH      3
Name: match_group_id, dtype: int64
cardinality
1:1    209
1:0     30
N:1     11
0:1     11
1:N      8
N:0      4
N:M      1
Name: count, dtype: int64


## 6. Task 2 - reconcile June, carrying May's backlog

June is not reconciled on its own. Everything still open at 31 May is added to the June
pool, so a May transaction can be cleared by a June credit. In the data these arrive with
narrations like `SETTLEMENT PRMAY59755172 PREV CYCLE`.

The `origin` column on every row records whether the transaction came from May or June, so
the three things the assignment asks for are all readable from one file:

1. June transactions settled in June - `origin = JUN`, status `MATCHED`.
2. May backlog cleared in June - `origin = MAY`, status `MATCHED`.
3. Items still open at 30 June - status `OPEN`.

In [6]:
may_open_ids = []
for g in may_groups:
    if g["status"] == "OPEN":
        may_open_ids.extend(g["internal_txn_ids"].split(","))

backlog_txns = may_internal[may_internal["txn_id"].isin(may_open_ids)].copy()
backlog_txns["origin_period"] = "MAY"
jun_internal["origin_period"] = "JUN"

jun_scope = pd.concat([jun_internal, backlog_txns], ignore_index=True)
jun_groups = reconcile(jun_scope, jun_bank, "JUN", JUN_END)
jun_out = pd.DataFrame(jun_groups)
print("May backlog carried into June:", len(backlog_txns))
print(jun_out.groupby("status")["match_group_id"].count())
print(jun_out["cardinality"].value_counts())

May backlog carried into June: 30
status
EXCEPTION         12
MATCHED          228
OPEN              32
PARTIAL_MATCH      2
Name: match_group_id, dtype: int64
cardinality
1:1    214
1:0     32
N:1      9
0:1      9
1:N      6
N:0      3
N:M      1
Name: count, dtype: int64


## 7. Settlement lag or a genuine exception, then write the main file

**The rule: an item still open at month end is a *Likely Settlement Lag* if it is 5 days old
or less, and a *Genuine Exception* if it is older.** These are the two labels the brief asks
for, used exactly as written.

Five days covers a normal settlement cycle plus a weekend. The data agrees: of May's 30 open
items, the 22 that were 5 days old or less at 31 May **all** settled in June, and the 8 older
ones did not.

This cell also sets the plain English words used by every output file, then writes
`reconciliation_output.csv` with 13 easy columns instead of the 21 internal ones. The status
column keeps the brief's own words (Matched / Partial Match / Open / Exception) and the
cardinality column keeps 1:1 / 1:N / N:1 / N:M with a short gloss next to each.

In [ ]:
all_out = pd.concat([may_out, jun_out], ignore_index=True)


def open_classification(row):
    if row["status"] != "OPEN":
        return ""
    if row["ageing_days"] <= LAG_DAYS:
        return "Likely Settlement Lag"
    return "Genuine Exception"


all_out["open_item_classification"] = all_out.apply(open_classification, axis=1)
all_out.loc[(all_out["status"] == "OPEN") &
            (all_out["open_item_classification"] == "Genuine Exception"),
            "exception_category"] = "unsettled_transaction"

# ---------------------------------------------------------------------
# Plain English words. The engine uses short codes; every file we hand
# over uses these words instead, so nobody has to look up a code.
# ---------------------------------------------------------------------
MONTH_WORDS = {"MAY": "May 2026", "JUN": "June 2026"}

# The brief requires these four exact statuses. We keep the required word,
# title-cased so it still reads as plain English.
RESULT_WORDS = {
    "MATCHED": "Matched",
    "PARTIAL_MATCH": "Partial Match",
    "OPEN": "Open",
    "EXCEPTION": "Exception",
}

# The brief requires the 1:1 / 1:N / N:1 / N:M labels. We keep the label and
# add a plain gloss next to it, instead of replacing it.
MATCH_TYPE_WORDS = {
    "1:1": "1:1 - one payment, one credit",
    "1:N": "1:N - one payment, many credits",
    "N:1": "N:1 - many payments, one credit",
    "N:M": "N:M - many payments, many credits",
    "1:0": "1:0 - payment with no credit",
    "0:1": "0:1 - credit with no payment",
    "N:0": "N:0 - entries that cancel out",
}

# These five phrases are the exception categories named in the brief. We use
# them exactly as written, not a paraphrase, so they are easy to search for.
PROBLEM_WORDS = {
    "duplicate_credit": "Duplicate credit",
    "short_settlement": "Short settlement",
    "orphan_credit": "Orphan credit",
    "unsettled_transaction": "Unsettled transaction",
    "internal_self_netting_pair": "Internal self-netting pair",
}

# The engine's match_method is a numbered pass name ("4. payment reference").
# This turns it into one plain phrase, without dropping the field.
METHOD_WORDS = {
    "1. internal self-netting": "Internal netting (sale + reversal)",
    "2. batch id": "Batch settlement",
    "3. part settlement": "Part settlement",
    "4. payment reference": "Payment reference match",
    "5. net settlement sweep": "Net settlement sweep",
    "6. unmatched": "No match found",
}

# ---- the main file: 13 easy columns instead of 21 code-like ones -----
simple_output = pd.DataFrame({
    "Month": all_out["period"].map(MONTH_WORDS),
    "Group ID": all_out["match_group_id"],
    "Our Transaction IDs": all_out["internal_txn_ids"],
    "Bank Line IDs": all_out["bank_line_ids"],
    "Payment Reference": all_out["payment_refs"],
    "Cardinality": all_out["cardinality"].map(MATCH_TYPE_WORDS),
    "Our Amount": all_out["internal_amount"],
    "Bank Amount": all_out["bank_amount"],
    "Difference": all_out["variance"],
    "Status": all_out["status"].map(RESULT_WORDS),
    "Days Old": all_out["ageing_days"],
    "Match Method": all_out["match_method"].map(METHOD_WORDS),
    "Comments": all_out["comments"],
})
simple_output.to_csv(os.path.join(OUT_DIR, "reconciliation_output.csv"), index=False)
print(simple_output["Status"].value_counts().to_string())

## 8. Task 3 - the problem list

Every problem gets one short label and one short instruction. Eleven columns.

In [ ]:
ACTIONS = {
    "duplicate_credit": "Ask the bank to take the extra credit back.",
    "short_settlement": "Recover the shortfall. It is usually a bank fee.",
    "orphan_credit": "Ask the bank whose money this is.",
    "unsettled_transaction": "Chase the acquirer. This payment is overdue.",
    "internal_self_netting_pair": "No action needed. Keep for the audit trail.",
}

exceptions = all_out[all_out["exception_category"] != ""].copy()


def exception_amount(row):
    if row["internal_amount"] != 0:
        return row["internal_amount"]
    if row["bank_amount"] != 0:
        return row["bank_amount"]
    return row["internal_gross_amount"]      # self-netting pairs net to zero


exceptions["amount"] = exceptions.apply(exception_amount, axis=1)
exceptions["recommended_action"] = exceptions["exception_category"].map(ACTIONS)

# ---- 11 easy columns instead of 18 ----------------------------------
simple_exceptions = pd.DataFrame({
    "Month": exceptions["period"].map(MONTH_WORDS),
    "Group ID": exceptions["match_group_id"],
    "Problem": exceptions["exception_category"].map(PROBLEM_WORDS),
    "Our Transaction IDs": exceptions["internal_txn_ids"],
    "Bank Line IDs": exceptions["bank_line_ids"],
    "Payment Reference": exceptions["payment_refs"],
    "Amount": exceptions["amount"],
    "Difference": exceptions["variance"],
    "Days Old": exceptions["ageing_days"],
    "What Happened": exceptions["comments"],
    "What To Do": exceptions["recommended_action"],
})
simple_exceptions.to_csv(os.path.join(OUT_DIR, "exception_report.csv"), index=False)
print(simple_exceptions.groupby("Problem").agg(Items=("Group ID", "count"),
                                               Amount=("Amount", lambda s: round(s.abs().sum(), 2))))

## 9. Backlog report

One row per May item that was still open at 31 May, showing whether June cleared it, and
classifying anything still open as Likely Settlement Lag or Genuine Exception. Nine columns.

In [ ]:
jun_txn_to_group = {}
for g in jun_groups:
    if g["internal_txn_ids"]:
        for t in g["internal_txn_ids"].split(","):
            jun_txn_to_group[t] = g

backlog_rows = []
for _, t in backlog_txns.iterrows():
    g = jun_txn_to_group.get(t["txn_id"])
    cleared = g is not None and g["status"] in ("MATCHED", "PARTIAL_MATCH")
    age_jun = (JUN_END - t["txn_date"]).days
    if cleared:
        classification = "Cleared in June"
        note = "June credit " + g["bank_line_ids"] + " settled this."
    elif age_jun <= LAG_DAYS:
        classification = "Likely Settlement Lag"
        note = "Still open on 30 June. Normal delay, not old enough to worry about yet."
    else:
        classification = "Genuine Exception"
        note = "Still open on 30 June. Past the normal delay window. Chase the bank."
    backlog_rows.append({
        "txn_id": t["txn_id"], "payment_ref": t["payment_ref"], "merchant_id": t["merchant_id"],
        "txn_date": str(t["txn_date"].date()),
        "amount": round(float(t["amount"]), 2),
        "cleared_in_june": "Yes" if cleared else "No",
        "ageing_days_at_june_end": age_jun,
        "classification": classification,
        "comments": note,
    })
backlog = pd.DataFrame(backlog_rows)

# ---- 9 easy columns instead of 18 -----------------------------------
simple_backlog = pd.DataFrame({
    "Transaction ID": backlog["txn_id"],
    "Payment Reference": backlog["payment_ref"],
    "Merchant": backlog["merchant_id"],
    "Payment Date": backlog["txn_date"],
    "Amount": backlog["amount"],
    "Cleared In June": backlog["cleared_in_june"],
    "Days Old On 30 June": backlog["ageing_days_at_june_end"],
    "Classification": backlog["classification"],
    "Comments": backlog["comments"],
})
simple_backlog.to_csv(os.path.join(OUT_DIR, "backlog_report.csv"), index=False)
print(simple_backlog["Classification"].value_counts().to_string())

## 10. Summary report

The headline numbers for both months, in four columns: month, what we counted, how many, amount.

In [ ]:
summary_rows = []


def add_summary(period, metric, count, value):
    summary_rows.append({"period": period, "metric": metric,
                         "count": count, "value": round(value, 2)})


for period, internal_df, bank_df in [("MAY", may_internal, may_bank),
                                     ("JUN", jun_scope, jun_bank)]:
    part = all_out[all_out["period"] == period]
    add_summary(period, "Payments in our books", len(internal_df), internal_df["amount"].sum())
    add_summary(period, "Credits on the bank statement", len(bank_df), bank_df["amount"].sum())
    for st in ["MATCHED", "PARTIAL_MATCH", "OPEN", "EXCEPTION"]:
        sub = part[part["status"] == st]
        value = sub.apply(exception_amount, axis=1).abs().sum() if len(sub) > 0 else 0.0
        add_summary(period, RESULT_WORDS[st], len(sub), value)
    settled = part[part["status"].isin(["MATCHED", "PARTIAL_MATCH"])]
    add_summary(period, "Groups settled with a difference",
                len(settled[settled["variance"].abs() > TOLERANCE]), settled["variance"].sum())

add_summary("MAY", "May items carried into June", len(backlog_txns), backlog_txns["amount"].sum())
cleared = backlog[backlog["cleared_in_june"] == "Yes"]
still_open = backlog[backlog["cleared_in_june"] == "No"]
add_summary("JUN", "May items cleared in June", len(cleared), cleared["amount"].sum())
add_summary("JUN", "May items still open", len(still_open), still_open["amount"].sum())

jun_open = all_out[(all_out["period"] == "JUN") & (all_out["status"] == "OPEN")]
lag = jun_open[jun_open["open_item_classification"] == "Likely Settlement Lag"]
genuine = jun_open[jun_open["open_item_classification"] == "Genuine Exception"]
add_summary("JUN", "Open - normal delay", len(lag), lag["internal_amount"].sum())
add_summary("JUN", "Open - needs checking", len(genuine), genuine["internal_amount"].sum())

summary = pd.DataFrame(summary_rows)

# ---- 4 easy columns --------------------------------------------------
simple_summary = pd.DataFrame({
    "Month": summary["period"].map(MONTH_WORDS),
    "What We Counted": summary["metric"],
    "How Many": summary["count"],
    "Amount": summary["value"],
})
simple_summary.to_csv(os.path.join(OUT_DIR, "summary_control_report.csv"), index=False)
print(simple_summary.to_string(index=False))

## 11. Input check report

Every check we ran on the four input files, written out in plain words.

In [ ]:
CHECK_WORDS = {
    "FILE_READABLE": "File opens correctly",
    "REQUIRED_COLUMNS": "All needed columns are there",
    "VALID_DATE": "Dates are real dates",
    "VALID_AMOUNT": "Amounts are numbers",
    "ID_PRESENT": "Every row has an ID",
    "UNIQUE_ID": "No ID is repeated",
    "PAYMENT_REF_PRESENT": "Every row has a payment reference",
    "CURRENCY": "Currency is INR",
    "STATUS": "Only successful payments are used",
    "DUPLICATE_PAYMENT_REF": "Payment reference used more than once",
    "CREDIT_ONLY": "Only money-in lines are used",
}

RESULT_OF_CHECK = {"PASS": "OK", "WARNING": "Warning", "ERROR": "Error"}

validation = pd.DataFrame(validation_rows)

# ---- 5 easy columns --------------------------------------------------
simple_validation = pd.DataFrame({
    "File": validation["file"],
    "What We Checked": validation["check"].map(CHECK_WORDS),
    "Result": validation["severity"].map(RESULT_OF_CHECK),
    "Row ID": validation["record_id"],
    "Details": validation["issue_detail"],
})
simple_validation.to_csv(os.path.join(OUT_DIR, "input_validation_report.csv"), index=False)
print(simple_validation["Result"].value_counts().to_string())

## 12. Add-on - the HTML dashboard

Built straight from the tables above by joining strings together, so the dashboard can never
disagree with the CSV files. No charting library and no internet needed - the file opens in
any browser and is styled with plain CSS.

In [ ]:
def money(x):
    return "{:,.2f}".format(round(float(x), 2))


BADGE = {
    "Matched": "ok",
    "Partial Match": "warn",
    "Open": "wait",
    "Exception": "bad",
    "Cleared in June": "ok",
    "Likely Settlement Lag": "wait",
    "Genuine Exception": "bad",
}


def looks_like_a_number(text):
    """True if the text is a figure, so we can line the digits up on the right."""
    for ch in text:
        if ch.isalpha():
            return False
    return True


def table(headers, rows):
    """Column 1 is always text. The rest follow whatever the first row looks like."""
    aligns = []
    for i in range(len(headers)):
        figure = i > 0 and len(rows) > 0 and looks_like_a_number(str(rows[0][i]))
        aligns.append("num" if figure else "txt")

    html = "<table><thead><tr>"
    for i, h in enumerate(headers):
        html += '<th class="' + aligns[i] + '">' + h + "</th>"
    html += "</tr></thead><tbody>"
    for r in rows:
        html += "<tr>"
        for i, c in enumerate(r):
            text = str(c)
            if i == 0 and text in BADGE:
                html += ('<td class="txt"><span class="badge ' + BADGE[text] + '">'
                         + text + "</span></td>")
            else:
                html += '<td class="' + aligns[i] + '">' + text + "</td>"
        html += "</tr>"
    return html + "</tbody></table>"


def kpi(label, big, small):
    return ('<div class="kpi"><div class="kpi-label">' + label + "</div>"
            '<div class="kpi-value">' + big + "</div>"
            '<div class="kpi-note">' + small + "</div></div>")


def status_rows(period):
    part = all_out[all_out["period"] == period]
    rows = []
    for st in ["MATCHED", "PARTIAL_MATCH", "OPEN", "EXCEPTION"]:
        sub = part[part["status"] == st]
        value = sub.apply(exception_amount, axis=1).abs().sum()
        share = 0 if len(part) == 0 else round(100.0 * len(sub) / len(part), 1)
        rows.append([RESULT_WORDS[st], len(sub), money(value), str(share) + " %"])
    rows.append(["Total", len(part), "", "100.0 %"])
    return rows


def match_type_rows(period):
    part = all_out[all_out["period"] == period]
    rows = []
    for c in ["1:1", "1:N", "N:1", "N:M", "1:0", "0:1", "N:0"]:
        sub = part[part["cardinality"] == c]
        if len(sub) > 0:
            rows.append([MATCH_TYPE_WORDS[c], len(sub),
                         money(sub.apply(exception_amount, axis=1).abs().sum())])
    return rows


problem_rows = []
for cat in sorted(exceptions["exception_category"].unique()):
    sub = exceptions[exceptions["exception_category"] == cat]
    problem_rows.append([PROBLEM_WORDS[cat], len(sub),
                         money(sub["amount"].abs().sum()), ACTIONS[cat]])

AGE_WORDS = {"0-7d": "Up to 7 days", "8-15d": "8 to 15 days", "16-30d": "16 to 30 days",
             "31-60d": "31 to 60 days", "60d+": "More than 60 days"}
ageing_rows = []
open_and_exception = all_out[all_out["status"].isin(["OPEN", "EXCEPTION"])]
for b in ["0-7d", "8-15d", "16-30d", "31-60d", "60d+"]:
    sub = open_and_exception[open_and_exception["ageing_bucket"] == b]
    if len(sub) > 0:
        ageing_rows.append([AGE_WORDS[b], len(sub),
                            money(sub.apply(exception_amount, axis=1).abs().sum())])

backlog_rows_html = [
    ["May items carried into June", len(backlog), money(backlog["amount"].sum())],
    ["Cleared in June", len(cleared), money(cleared["amount"].sum())],
    ["Still open on 30 June", len(still_open), money(still_open["amount"].sum())],
]

closing_rows = [
    ["Likely Settlement Lag", len(lag), money(lag["internal_amount"].abs().sum())],
    ["Genuine Exception", len(genuine), money(genuine["internal_amount"].abs().sum())],
]

total_groups = len(all_out)
matched_groups = len(all_out[all_out["status"] == "MATCHED"])
match_rate = round(100.0 * matched_groups / total_groups, 1)
problem_count = len(exceptions)
problem_value = money(exceptions["amount"].abs().sum())

CSS = """
:root {
  --ink: #1b2333;
  --muted: #667085;
  --line: #e3e7ee;
  --card: #ffffff;
  --page: #f5f7fb;
  --brand: #2f5fd8;
  --ok: #12805c;   --ok-bg: #e3f5ee;
  --warn: #9a6206; --warn-bg: #fdf1dc;
  --wait: #2f5fd8; --wait-bg: #e6edfd;
  --bad: #b42318;  --bad-bg: #fdeceb;
}
* { box-sizing: border-box; }
body {
  margin: 0; padding: 32px 20px 60px;
  background: var(--page); color: var(--ink);
  font-family: "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
  font-size: 14px; line-height: 1.55;
}
.wrap { max-width: 1020px; margin: 0 auto; }
header {
  background: linear-gradient(135deg, #2f5fd8 0%, #1b3f9e 100%);
  color: #fff; border-radius: 14px; padding: 26px 28px;
  box-shadow: 0 6px 18px rgba(27, 63, 158, 0.18);
}
header h1 { margin: 0 0 6px; font-size: 24px; font-weight: 600; letter-spacing: 0.2px; }
header p { margin: 0; font-size: 13px; opacity: 0.88; }
.kpis { display: flex; flex-wrap: wrap; gap: 14px; margin: 20px 0 6px; }
.kpi {
  flex: 1 1 150px; background: var(--card); border: 1px solid var(--line);
  border-radius: 12px; padding: 16px 18px;
}
.kpi-label { font-size: 12px; color: var(--muted); text-transform: uppercase; letter-spacing: 0.6px; }
.kpi-value { font-size: 26px; font-weight: 600; margin: 4px 0 2px; color: var(--brand); }
.kpi-note { font-size: 12px; color: var(--muted); }
section {
  background: var(--card); border: 1px solid var(--line); border-radius: 12px;
  padding: 20px 22px; margin-top: 18px;
}
section h2 {
  margin: 0 0 4px; font-size: 16px; font-weight: 600;
  display: flex; align-items: center; gap: 9px;
}
section h2 .n {
  background: var(--brand); color: #fff; border-radius: 6px;
  font-size: 12px; padding: 1px 8px; font-weight: 600;
}
.lead { color: var(--muted); font-size: 13px; margin: 0 0 12px; }
.two { display: flex; flex-wrap: wrap; gap: 18px; }
.two > div { flex: 1 1 380px; min-width: 300px; }
.sub { font-weight: 600; font-size: 13px; margin-bottom: 2px; }
table { border-collapse: collapse; width: 100%; margin-top: 8px; font-size: 13px; }
th, td { padding: 8px 10px; border-bottom: 1px solid var(--line); text-align: left; }
th {
  background: #f0f3f9; color: #48536b; font-weight: 600;
  font-size: 12px; text-transform: uppercase; letter-spacing: 0.4px;
}
.num { text-align: right; font-variant-numeric: tabular-nums; }
.txt { text-align: left; }
tbody tr:hover { background: #f8fafd; }
tbody tr:last-child td { border-bottom: none; }
.badge {
  display: inline-block; padding: 2px 10px; border-radius: 999px;
  font-size: 12px; font-weight: 600; white-space: nowrap;
}
.badge.ok { background: var(--ok-bg); color: var(--ok); }
.badge.warn { background: var(--warn-bg); color: var(--warn); }
.badge.wait { background: var(--wait-bg); color: var(--wait); }
.badge.bad { background: var(--bad-bg); color: var(--bad); }
.note {
  font-size: 12.5px; color: var(--muted); margin: 12px 0 0;
  border-left: 3px solid var(--line); padding-left: 12px;
}
footer { color: var(--muted); font-size: 12px; text-align: center; margin-top: 22px; }
@media (max-width: 700px) { .two > div { min-width: 100%; } }
"""

html = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Reconciliation Dashboard - May &amp; June 2026</title>
<style>""" + CSS + """</style>
</head>
<body>
<div class="wrap">

<header>
  <h1>Bank Settlement Reconciliation</h1>
  <p>May and June 2026 &middot; all amounts in Indian Rupees &middot; built on """ + str(pd.Timestamp.today().date()) + """</p>
</header>

<div class="kpis">
""" + kpi("Groups checked", str(total_groups), "Both months together") + """
""" + kpi("Matched", str(match_rate) + " %", str(matched_groups) + " groups tied out exactly") + """
""" + kpi("Problems to act on", str(problem_count), "Worth " + problem_value) + """
""" + kpi("Open on 30 June", str(len(jun_open)), "No bank credit yet") + """
</div>

<section>
  <h2><span class="n">1</span> How each month turned out</h2>
  <p class="lead">A group is one unit of work. It can be one payment and one credit, or several of either.</p>
  <div class="two">
    <div>
      <div class="sub">May 2026</div>
      <div class="lead">""" + str(len(may_internal)) + """ payments in our books vs """ + str(len(may_bank)) + """ bank credits</div>
      """ + table(["Status", "Groups", "Amount", "Share"], status_rows("MAY")) + """
      <p class="note">Our books """ + money(may_internal["amount"].sum()) + """ &middot; bank credits """ + money(may_bank["amount"].sum()) + """</p>
    </div>
    <div>
      <div class="sub">June 2026</div>
      <div class="lead">""" + str(len(jun_scope)) + """ payments (""" + str(len(backlog_txns)) + """ carried from May) vs """ + str(len(jun_bank)) + """ bank credits</div>
      """ + table(["Status", "Groups", "Amount", "Share"], status_rows("JUN")) + """
      <p class="note">Our books """ + money(jun_scope["amount"].sum()) + """ &middot; bank credits """ + money(jun_bank["amount"].sum()) + """</p>
    </div>
  </div>
  <p class="note">Amount is what our books say. Where our books have nothing, it is what the bank paid.</p>
</section>

<section>
  <h2><span class="n">2</span> How payments were matched</h2>
  <p class="lead">The bank does not always send one credit per payment. This is the mix we found.</p>
  <div class="two">
    <div><div class="sub">May 2026</div>""" + table(["Match type", "Groups", "Amount"], match_type_rows("MAY")) + """</div>
    <div><div class="sub">June 2026</div>""" + table(["Match type", "Groups", "Amount"], match_type_rows("JUN")) + """</div>
  </div>
</section>

<section>
  <h2><span class="n">3</span> May items carried into June</h2>
  <p class="lead">May payments with no bank credit by 31 May were added to the June run, to see if June cleared them.</p>
  """ + table(["Item", "How many", "Amount"], backlog_rows_html) + """
  <p class="note">Rule we used: an open item is a <b>Likely Settlement Lag</b> if it is """ + str(LAG_DAYS) + """ days old or less, and a <b>Genuine Exception</b> if it is older.
  All """ + str(len(still_open)) + """ May items still open on 30 June are well past that, so every one of them is a Genuine Exception.</p>
</section>

<section>
  <h2><span class="n">4</span> Problems found, and what to do</h2>
  <p class="lead">Both months together. One line per type of problem.</p>
  """ + table(["Problem", "Items", "Amount", "What to do"], problem_rows) + """
  <p class="note">A May item still unsettled on 30 June appears in the May close and again in the June close.
  The June rows are the closing position.</p>
</section>

<section>
  <h2><span class="n">5</span> How old the unfinished items are</h2>
  <p class="lead">Age is counted up to the end of the month the item belongs to.</p>
  """ + table(["Age", "Items", "Amount"], ageing_rows) + """
</section>

<section>
  <h2><span class="n">6</span> Where we stand on 30 June 2026</h2>
  <p class="lead">Payments still waiting for a bank credit at the June close.</p>
  """ + table(["Classification", "Items", "Amount"], closing_rows) + """
</section>

<footer>
  Built from reconciliation_output.csv, exception_report.csv, backlog_report.csv,
  summary_control_report.csv and input_validation_report.csv.
</footer>

</div>
</body>
</html>"""

with open(os.path.join(OUT_DIR, "reconciliation_dashboard.html"), "w", encoding="utf-8") as f:
    f.write(html)

print("Dashboard written to", os.path.join(OUT_DIR, "reconciliation_dashboard.html"))

## 13. Final checks

The last word goes to the control checks: every transaction and every bank credit appears
in exactly one group, with no repeats. If any of these fail the notebook raises an error.

In [13]:
print("Every record appears exactly once:")
for period, internal_df, bank_df in [("MAY", may_internal, may_bank), ("JUN", jun_scope, jun_bank)]:
    part = all_out[all_out["period"] == period]
    txn_ids = []
    line_ids = []
    for _, r in part.iterrows():
        if isinstance(r["internal_txn_ids"], str) and r["internal_txn_ids"]:
            txn_ids += r["internal_txn_ids"].split(",")
        if isinstance(r["bank_line_ids"], str) and r["bank_line_ids"]:
            line_ids += r["bank_line_ids"].split(",")
    assert len(txn_ids) == len(set(txn_ids)) == len(internal_df), period + " transaction count is wrong"
    assert len(line_ids) == len(set(line_ids)) == len(bank_df), period + " bank line count is wrong"
    print(" ", period, "-", len(txn_ids), "transactions and", len(line_ids), "bank credits, no repeats")

print("\nFiles written:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  " + f)

Every record appears exactly once:
  MAY - 328 transactions and 252 bank credits, no repeats
  JUN - 318 transactions and 250 bank credits, no repeats

Files written:
  backlog_report.csv
  exception_report.csv
  input_validation_report.csv
  reconciliation_dashboard.html
  reconciliation_output.csv
  summary_control_report.csv
